# Main.py

Ini buat Install Library

In [4]:
!pip install -qU \
    opentelemetry-api==1.38.0 \
    opentelemetry-sdk==1.38.0 \
    opentelemetry-exporter-otlp-proto-common==1.38.0 \
    opentelemetry-proto==1.38.0 \
    langchain-groq \
    langchain-community \
    langchain-chroma \
    langchain-huggingface \
    pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.5 MB/s eta 0:00:00


Ini Nginstall PDF Reader

In [ ]:
!pip install pypdf
!pip install langchain-community pypdf

Ini buat import database

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader('Dokumen_Tes_RAG_Rexaldo.pdf')
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts = text_splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(documents=texts, embedding=embeddings)

print("Database Dora the explorer berhasil dibuat yey >.<")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Database Dora the explorer berhasil dibuat yey >.<


Yang ini,eksekusi buat RAG

In [7]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Setup API
os.environ["GROQ_API_KEY"] = "RAHASIA GES"

# 2. Panggil LLM
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)

# 3. Setup Prompt
system_prompt = (
    "Kamu adalah asisten AI yang sangat cerdas.\n"
    "Gunakan data berikut untuk menjawab pertanyaan karyawan.\n"
    "Jika jawabannya tidak ada di data ini, bilang saja 'Data tidak ditemukan'.\n\n"
    "Konteks:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

# 4. Fungsi buat ngerapiin teks dari database
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 5. RAG CHAIN MURNI (Tanpa langchain.chains!)
retriever = vectorstore.as_retriever()

rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 6. Eksekusi
query = "Kalo semisalkan pegawai ada yang telat 20 menit setelah jam kerja,apa yang didapat?"
response = rag_chain.invoke(query)

print(f"Hasil AI: {response}")

Hasil AI: Menurut data yang saya miliki, karyawan yang datang terlambat lebih dari 15 menit akan dikenakan potongan uang makan sebesar Rp 25.000.

Jadi, jika karyawan datang terlambat 20 menit, maka mereka akan dikenakan potongan uang makan sebesar Rp 25.000.


Yang ini buat output pertanyaan sama hasil

In [8]:
pertanyaan_baru = "Gimana cara dapat tunjangan?"
output = rag_chain.invoke(pertanyaan_baru)

print(f"Pertanyaan: {pertanyaan_baru}")
print(f"Jawaban: {output}")

Pertanyaan: Gimana cara dapat tunjangan?
Jawaban: Untuk mendapatkan tunjangan tambahan Rp 1.000.000/bulan, kamu harus menyelesaikan sertifikasi cloud (AWS/Azure/GCP).
